# 03 — Upload Processed Parquet to S3 (Silver Layer)

**Goal:** Push the London-filtered Parquet file from local disk to S3's `staging/parquet/` location. This is the **Silver layer** — analytics-ready data that Snowflake will ingest.

**Why this layer exists separately from Bronze:**
- Bronze (raw CSVs): full England & Wales dataset, 600MB
- Silver (processed Parquet): London-only, columnar, compressed, 5.7MB
- Separation makes the architecture explicit: raw vs. refined

**Path:** `data/processed/london_transactions_raw.parquet` → `s3://london-housing-analytics-irfan/staging/parquet/`

## Imports

In [1]:
import os
import boto3
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(PROJECT_ROOT / '.env')

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

bucket = os.getenv('S3_BUCKET')
print(f"✅ S3 client ready — bucket: {bucket}")

✅ S3 client ready — bucket: london-housing-analytics-irfan


## Upload Parquet

In [2]:
local_path = PROJECT_ROOT / 'data' / 'processed' / 'london_transactions_raw.parquet'
s3_key = 'staging/parquet/london_transactions_raw.parquet'

file_size_mb = local_path.stat().st_size / (1024 * 1024)
print(f"📤 Uploading {local_path.name} ({file_size_mb:.1f} MB) to Silver layer...")

s3.upload_file(
    Filename=str(local_path),
    Bucket=bucket,
    Key=s3_key
)

print(f"Upload complete → s3://{bucket}/{s3_key}")

📤 Uploading london_transactions_raw.parquet (5.7 MB) to Silver layer...
Upload complete → s3://london-housing-analytics-irfan/staging/parquet/london_transactions_raw.parquet


## Verify

In [3]:
response = s3.list_objects_v2(Bucket=bucket, Prefix='staging/parquet/')

print(f"Files in s3://{bucket}/staging/parquet/ (Silver layer):\n")
for obj in response.get('Contents', []):
    size_mb = obj['Size'] / (1024 * 1024)
    print(f"  - {obj['Key']} ({size_mb:.1f} MB)")

Files in s3://london-housing-analytics-irfan/staging/parquet/ (Silver layer):

  - staging/parquet/ (0.0 MB)
  - staging/parquet/london_transactions_raw.parquet (5.7 MB)
